In [1]:
try:
    print_spark_info()
except:
    %run ../spark-instance.ipynb

SparkConf created
Started SparkSession
Spark version 3.5.3
You should be able to access the Spark UI at: https://dacs-compute-gate.ewi.utwente.nl:9999/user/g.luvizottocesar@utwente.nl/proxy/4040/stages/
Note that you may have to Enable extensions first via the Extension Manager.


In [4]:
clean_spark()

CLEANING SPARK INSTANCE...


In [2]:
import os
import logging
from datetime import datetime

# installed pkg imports
import numpy as np
import pandas as pd

# pyspark related imports
import pyspark.sql.types as pst
from pyspark.sql import functions as psf
from pyspark.sql.window import Window

In [3]:
# Create logger
logger = logging.getLogger("anycast_services")
logger.setLevel(logging.INFO)

# Prevent duplicate logs if logger already has handlers
if not logger.handlers:
    # Create file handler
    file_handler = logging.FileHandler("logs.log", encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    
    # Create formatter
    formatter = logging.Formatter(
        fmt='%(asctime)s %(levelname)-8s %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Add formatter to handler
    file_handler.setFormatter(formatter)
    
    # Add handler to logger
    logger.addHandler(file_handler)
else:
    print("logger already exist!")

# Optionally, prevent propagation to root logger to avoid duplicates
logger.propagate = False

## LZR

In [4]:
base_path = "s3a://catrin/measurements/tool=lzr/dataset=tcp-anycast/format={f}/vp={vp}/port={port}"

In [5]:
def convert_lzr(vp, dates, ports):
    logger.info(f"convert vp {vp}")
    for d in dates:
        logger.info(f"convert date {d}")
        for port in ports:
            if port in [53, 123, 853]:  # no lzr scan on those ports
                logger.info(f"port {port} was not scanned with lzr")
                continue
            logger.info(f"port {port}")
            f = "raw"
            raw_base_path = base_path.format(f=f, port=port, vp=vp)
            raw_full_path = os.path.join(raw_base_path, f"year={d.year}/month={d.month:02d}/day={d.day:02d}")
            lzr_df = spark.read.option("basePath", raw_base_path).json(raw_full_path)
            logger.info("LZR " + str(datetime.now()) + f" - read raw")
            
            f = "parquet"
            parquet_base_path = base_path.format(f=f, port=port, vp=vp)
            parquet_full_path = os.path.join(parquet_base_path, f"year={d.year}/month={d.month:02d}/day={d.day:02d}")
            lzr_df.write.option("basePath", parquet_base_path).parquet(parquet_full_path)
            logger.info("LZR " + str(datetime.now()) + f" - wrote parquet")

In [14]:
dates = [
    datetime(2026, 3, 3),
]

vp = "au-syd"
ds = "tcp-anycast"

In [7]:
ports_pdf = pd.read_csv("lzr_ports.txt", header=None)
ports_pdf.columns = ["port"]
print(ports_pdf)
ports = ports_pdf["port"].to_list()

      port
0      123
1       53
2      853
3    32768
4      513
..     ...
115   5101
116   1521
117   6646
118    502
119   5631

[120 rows x 1 columns]


In [11]:
if False:
    spark.read.option("basePath", f"s3a://catrin/measurements/tool=lzr/dataset={ds}/format=raw/vp={vp}/").json(f"s3a://catrin/measurements/tool=lzr/dataset={ds}/format=raw/vp={vp}/"
    ).filter(
        (psf.col("year") == dates[-1].year)
        & (psf.col("month") == dates[-1].month)
        & (psf.col("day") == dates[-1].day)
    ).select("port").distinct().count()

In [12]:
#spark.read.option("basePath", f"s3a://catrin/measurements/tool=lzr/dataset={ds}/format=raw/vp={vp}/").json(f"s3a://catrin/measurements/tool=lzr/dataset={ds}/format=raw/vp={vp}/"
#).filter(psf.col("day") == dates[0].day).groupBy("port").count().show()

In [15]:
convert_lzr(vp, dates, ports)

In [16]:
print("test")

test


## ZGrab

In [4]:
vps = ["nl-ens", "au-syd", "de-mun"]
datasets = ["tcp-anycast"]
ports = ["all_ports", "ssh_ports"]
snapshot = datetime(2026, 3, 4)
zgrab_base_path = "s3a://catrin/measurements/tool=zgrab/dataset={ds}/format={f}/vp={vp}/port={port}/year={year}/month={month:02d}/day={day:02d}"

def convert_zgrab(vps, datasets, snapshot):
    for port in ports:
        logger.info(f"{port}")
        print(f"{port}")
        for vp in vps:
            logger.info(f"{vp}")
            print(f"{vp}")
            for ds in datasets:
                logger.info(f"{ds}")
                print(f"{ds}")
                f = "raw"
                raw_path = zgrab_base_path.format(ds=ds, f=f, vp=vp, port=port, year=snapshot.year, month=snapshot.month, day=snapshot.day)
                zgrab_df = spark.read.json(raw_path)
                logger.info("ZGrab " + str(datetime.now()) + f" - read raw")
                print("ZGrab " + str(datetime.now()) + f" - read raw")
    
                f = "parquet"
                parquet_path = zgrab_base_path.format(ds=ds, f=f, vp=vp, port=port, year=snapshot.year, month=snapshot.month, day=snapshot.day)
                zgrab_df.write.parquet(parquet_path)
                logger.info("ZGrab " + str(datetime.now()) + f" - wrote parquet")
                print("ZGrab " + str(datetime.now()) + f" - wrote parquet")

In [5]:
convert_zgrab(vps, datasets, snapshot)

all_ports
nl-ens
au-syd
tcp-anycast
ZGrab 2026-03-06 18:52:58.024079 - read raw
ZGrab 2026-03-06 18:58:32.440748 - wrote parquet
de-mun
tcp-anycast
ZGrab 2026-03-06 19:01:04.471670 - read raw
ZGrab 2026-03-06 19:04:57.657767 - wrote parquet
ssh_ports
nl-ens
tcp-anycast
ZGrab 2026-03-06 19:04:58.037513 - read raw
ZGrab 2026-03-06 19:04:59.104696 - wrote parquet
au-syd
tcp-anycast
ZGrab 2026-03-06 19:04:59.510650 - read raw
ZGrab 2026-03-06 19:05:00.727718 - wrote parquet
de-mun
tcp-anycast
ZGrab 2026-03-06 19:05:01.029995 - read raw
ZGrab 2026-03-06 19:05:02.157645 - wrote parquet


### old zgrab conversion

In [8]:
# IPv4 NL no HRP
dataset = "tcp-anycast"
vp = "nl-ens"

# read jsonl
zgrab_v4_nl_ts = datetime(2025, 12, 14)
zgrab_v4_nl_df = spark.read.json(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=raw/vp={vp}/port=top100/year={zgrab_v4_nl_ts.year}/month={zgrab_v4_nl_ts.month:02d}/day={zgrab_v4_nl_ts.day:02d}/")
logger.info(str(datetime.now()) + f" read raw {dataset} from {vp}")

# write parquet
zgrab_v4_nl_df.write.parquet(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=parquet/vp={vp}/port=top100/year={zgrab_v4_nl_ts.year}/month={zgrab_v4_nl_ts.month:02d}/day={zgrab_v4_nl_ts.day:02d}/")
logger.info(str(datetime.now()) + f" wrote parquet {dataset} from {vp}")

In [7]:
# IPv4 NL HRP
dataset = "tcp-anycast-hrp"
vp = "nl-ens"

# read jsonl
zgrab_hrp_v4_nl_ts = datetime(2025, 12, 14)
zgrab_hrp_v4_nl_df = spark.read.json(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=raw/vp={vp}/port=top100/year={zgrab_hrp_v4_nl_ts.year}/month={zgrab_hrp_v4_nl_ts.month:02d}/day={zgrab_hrp_v4_nl_ts.day:02d}/")
logger.info(str(datetime.now()) + f" read raw {dataset} from {vp}")

# write parquet
zgrab_hrp_v4_nl_df.write.parquet(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=parquet/vp={vp}/port=top100/year={zgrab_hrp_v4_nl_ts.year}/month={zgrab_hrp_v4_nl_ts.month:02d}/day={zgrab_hrp_v4_nl_ts.day:02d}/")
logger.info(str(datetime.now()) + f" wrote parquet {dataset} from {vp}")

In [4]:
# IPv4 AU no HRP
dataset = "tcp-anycast"
vp = "au-syd"

# read jsonl
zgrab_v4_au_ts = datetime(2025, 12, 15)
zgrab_v4_au_df = spark.read.json(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=raw/vp={vp}/port=top100/year={zgrab_v4_au_ts.year}/month={zgrab_v4_au_ts.month:02d}/day={zgrab_v4_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" read raw {dataset} from {vp}")

# write parquet
zgrab_v4_au_df.write.parquet(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=parquet/vp={vp}/port=top100/year={zgrab_v4_au_ts.year}/month={zgrab_v4_au_ts.month:02d}/day={zgrab_v4_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" wrote parquet {dataset} from {vp}")

In [5]:
# IPv4 AU HRP
dataset = "tcp-anycast-hrp"
vp = "au-syd"

# read jsonl
zgrab_hrp_v4_au_ts = datetime(2025, 12, 16)
zgrab_hrp_v4_au_df = spark.read.json(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=raw/vp={vp}/port=top100/year={zgrab_hrp_v4_au_ts.year}/month={zgrab_hrp_v4_au_ts.month:02d}/day={zgrab_hrp_v4_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" read raw {dataset} from {vp}")

# write parquet
zgrab_hrp_v4_au_df.write.parquet(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=parquet/vp={vp}/port=top100/year={zgrab_hrp_v4_au_ts.year}/month={zgrab_hrp_v4_au_ts.month:02d}/day={zgrab_hrp_v4_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" wrote parquet {dataset} from {vp}")

In [6]:
# IPv6 AU
dataset = "tcp-anycast-v6"
vp = "au-syd"

# read jsonl
zgrab_v6_au_ts = datetime(2025, 12, 16)
zgrab_v6_au_df = spark.read.json(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=raw/vp={vp}/port=top100/year={zgrab_v6_au_ts.year}/month={zgrab_v6_au_ts.month:02d}/day={zgrab_v6_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" read raw {dataset} from {vp}")

# write parquet
zgrab_v6_au_df.write.parquet(f"s3a://catrin/measurements/tool=zgrab/dataset={dataset}/format=parquet/vp={vp}/port=top100/year={zgrab_v6_au_ts.year}/month={zgrab_v6_au_ts.month:02d}/day={zgrab_v6_au_ts.day:02d}/")
logger.info(str(datetime.now()) + f" wrote parquet {dataset} from {vp}")

In [ ]:
# IPv6 NL
# missing measurements